In [1]:
import sys
sys.path.append('/home/matin/McMaster/trimba/phd_codes/CTRNN')

In [9]:
import numpy as np
from dataset_generator.data_generator import *
from utils.other_utils import *
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from model.CTRNN import *
from easydict import EasyDict as edict
import yaml
from utils.train_helper import load_model
from sklearn.decomposition import PCA
import plotly
import plotly.graph_objs as go
import time
import colorsys
from sklearn.manifold import TSNE

# color range creator

In [3]:
def hsv2rgb(h,s,v):
    return tuple(round(i * 255) for i in colorsys.hsv_to_rgb(h,s,v))


def hsv_creator(num, span_percentage):
    span = np.linspace(250, 250 * (1 - span_percentage), num) / 360
    return [f'rgb{hsv2rgb(i, 1, 1)}' for i in span]

# Creating data

In [11]:
# data
data_duration = 35
dt  = 0.01
f_beat_time = 4
interval = 0.8
n_of_targets = 10
n_of_beats = 10
target_shape = 'full_sine'
data_x, data_y, first_beat_idx, _, _ = beats_chain_forced_generator(number_of_beats=n_of_beats,
                                                                    number_of_targets=n_of_targets,
                                                                    target_starts_from_beat=2,
                                                                    target_shape=target_shape,
                                                                    data_duration=data_duration,
                                                                    first_beat_time=f_beat_time,
                                                                    interval=interval,
                                                                    target_amplitude=1,
                                                                    beats_amplitude=1,
                                                                    dt=dt)

# find important points

In [12]:
def beat_chain_important_points(number_of_beats=10,
                                 data_duration=50,
                                 first_beat_time=4,
                                 interval=0.4,
                                 dt=dt):

    assert first_beat_time + (number_of_beats + 1) * interval < data_duration

    data_size = int(data_duration / dt)

    # create data_x
    data_x = torch.zeros((1, data_size))
    first_beat_idx = int(first_beat_time / dt)
    return [int((first_beat_time + i * interval) / dt) for i in range(number_of_beats)]

In [13]:
important_points = beat_chain_important_points(number_of_beats=n_of_beats, data_duration=data_duration,
                                               first_beat_time=f_beat_time, interval=interval, dt=dt)

# model

In [14]:
# model
config_file_path = '/home/matin/McMaster/trimba/phd_codes/CTRNN/experiments/ctrnn_v2/beats_chain/CTRNN_beats_chain_full_sine_beatsn10_targetsn10_dt0.01_2023-Feb-13-00-20-33_2965118/config.yaml'
config = edict(yaml.full_load(open(config_file_path, 'r')))
config.device = 'cpu'
model = CTRNN(config)
model_file = os.path.join(config.save_dir, config.test.test_model_name)
model_file = model_file.replace('/home/mtnusf97/projects/def-cannoj9/mtnusf97/CTRNN/', '../')
load_model(model, model_file, config.device)
model.to(config.device)

CTRNN(
  (wI): Linear(in_features=1, out_features=500, bias=True)
  (wR): Linear(in_features=500, out_features=500, bias=False)
  (wO): Linear(in_features=500, out_features=1, bias=True)
)

# predict

In [15]:
activations, hidden_states, outputs = model.init_activations_outputs(batch_size=1)
outputs, all_activations, all_hidden_states = model.detailed_predict(data_x,
                                                             activations.to(config.device),
                                                             hidden_states.to(config.device),
                                                             outputs)

all_activations = np.array([i.flatten().detach().numpy() for i in all_activations])
all_hidden_states = np.array([i.flatten().detach().numpy() for i in all_hidden_states])

# tSNE

In [23]:
tsne = TSNE(n_components=3, learning_rate='auto', init='random', perplexity=30)
activations_reduced_tsne = tsne.fit_transform(all_activations)

# PCA

In [22]:
pca = PCA(n_components=3)
pca.fit(all_activations)
print('it covers: ',np.sum(pca.explained_variance_ratio_) * 100, ' percent')
activations_reduced_pca = pca.transform(all_activations)

it covers:  90.12206196784973  percent


# static plot

In [24]:
symbols = np.array(['circle'] * len(activations_reduced))
symbols[important_points] = 'x'
text = np.array([None] * 3500)
text[important_points] = [f'beat_{i}' for i,_ in enumerate(important_points)]

In [ ]:
# Configure Plotly to be rendered inline in the notebook.
plotly.offline.init_notebook_mode()

# Configure the trace.
trace = go.Scatter3d(
    x=activations_reduced[:,0],  # <-- Put your data instead
    y=activations_reduced[:,1],  # <-- Put your data instead
    z=activations_reduced[:,2],  # <-- Put your data instead
    mode='markers+text',
    text=text,
    textposition='top center',
    marker={
        'size': 3,
        'opacity': 0.8,
        'color': hsv_creator(len(activations_reduced[:,0]), 1),
        'symbol': symbols
    }
)

# Configure the layout.
# layout = go.Layout(
#     margin={'l': 0, 'r': 0, 'b': 0, 't': 0}, width=1200, height=800
# )

layout = go.Layout(
    width=1200, height=800
)

data = [trace]

plot_figure = go.Figure(data=data, layout=layout)
# plot_figure = go.Figure(data=data)

# Render the plot.
# plotly.offline.iplot(plot_figure)
plot_figure.show()



In [139]:
plot_figure.write_html('interval1.2s_beats10.html')

# try animation

In [15]:
# Create figure
x=activations_reduced[:,0][:100]
y=activations_reduced[:,1][:100]
z=activations_reduced[:,2][:100]

In [16]:
data_0 = go.Scatter3d(x=[], y=[], z=[],
                      mode='markers',
                      marker={'size': 2,'opacity': 1,'color': 'red'})

In [17]:
layout = go.Layout(
    margin={'l': 0, 'r': 0, 'b': 0, 't': 0}
)

fig = go.Figure(data=[data_0], layout=layout)
#
# fig.update_layout(scene = dict(
#         xaxis=dict(range=[min(x), max(x)], autorange=False),
#         yaxis=dict(range=[min(y), max(y)], autorange=False),
#         zaxis=dict(range=[min(z), max(z)], autorange=False),
#         ))

In [18]:
frames = [go.Frame(data= [go.Scatter3d(
                                       x=x[:k+1],
                                       y=y[:k+1],
                                       z=z[:k+1])],

                   traces= [0],
                   name=f'frame{k}'
                  )for k  in  range(len(x)-1)]

# try animation

In [119]:
x = activations_reduced[:,0][:1000]
y = activations_reduced[:,1][:1000]
z = activations_reduced[:,2][:1000]

In [112]:
# Create figure
fig = go.Figure(go.Scatter3d(x=[], y=[], z=[],
                             mode="markers",
                             marker=dict(color='red', size=2)
                             )
                )

In [113]:
# Frames
data_len = len(x)
max_z = np.max(z)
frames = [go.Frame(data= [go.Scatter3d(x=x[:k+1],
                                       y=y[:k+1],
                                       z=z[:k+1],
                                       mode='markers',
                                       marker={'size': 2, 'opacity': 1, 
                                               'color': hsv_creator(k+1, (k+1)/data_len)}
                                       )
                          ],
                   traces= [0],
                   name=f'frame{k}'
                  )for k  in  range(len(x)-1)
          ]

In [ ]:
fig.update(frames=frames)

In [115]:
def frame_args(duration):
    return {
            "frame": {"duration": duration},
            "mode": "immediate",
            "fromcurrent": True,
            "transition": {"duration": duration, "easing": "linear"},
            }

In [116]:
sliders = [
    {"pad": {"b": 10, "t": 60},
     "len": 0.9,
     "x": 0.1,
     "y": 0,

     "steps": [
                 {"args": [[f.name], frame_args(100)],
                  "label": str(k),
                  "method": "animate",
                  } for k, f in enumerate(fig.frames)
              ]
     }
        ]

In [ ]:
fig.update_layout(
    updatemenus = [{"buttons":[
                    {
                        "args": [None, frame_args(5)],
                        "label": "Play",
                        "method": "animate",
                    },
                    {
                        "args": [[None], frame_args(0)],
                        "label": "Pause",
                        "method": "animate",
                  }],
                "font": {"color":"red"},
                "direction": "left",
                "pad": {"r": 10, "t": 70},
                "type": "buttons",
                "x": 0.1,
                "y": 0,
            }
         ],
         sliders=sliders
    )

In [ ]:
fig.update_layout(scene = dict(xaxis=dict(range=[min(x), max(x)], autorange=False),
                               yaxis=dict(range=[min(y), max(y)], autorange=False),
                               zaxis=dict(range=[min(z), max(z)], autorange=False)
                               )
                  )

# fig.update_layout(sliders=sliders)
fig.show()

In [91]:
hsv_creator(255, 1)

['rgb(42, 0, 255)',
 'rgb(38, 0, 255)',
 'rgb(34, 0, 255)',
 'rgb(30, 0, 255)',
 'rgb(26, 0, 255)',
 'rgb(22, 0, 255)',
 'rgb(17, 0, 255)',
 'rgb(13, 0, 255)',
 'rgb(9, 0, 255)',
 'rgb(5, 0, 255)',
 'rgb(1, 0, 255)',
 'rgb(0, 4, 255)',
 'rgb(0, 8, 255)',
 'rgb(0, 12, 255)',
 'rgb(0, 16, 255)',
 'rgb(0, 20, 255)',
 'rgb(0, 24, 255)',
 'rgb(0, 29, 255)',
 'rgb(0, 33, 255)',
 'rgb(0, 37, 255)',
 'rgb(0, 41, 255)',
 'rgb(0, 45, 255)',
 'rgb(0, 50, 255)',
 'rgb(0, 54, 255)',
 'rgb(0, 58, 255)',
 'rgb(0, 62, 255)',
 'rgb(0, 66, 255)',
 'rgb(0, 70, 255)',
 'rgb(0, 75, 255)',
 'rgb(0, 79, 255)',
 'rgb(0, 83, 255)',
 'rgb(0, 87, 255)',
 'rgb(0, 91, 255)',
 'rgb(0, 96, 255)',
 'rgb(0, 100, 255)',
 'rgb(0, 104, 255)',
 'rgb(0, 108, 255)',
 'rgb(0, 112, 255)',
 'rgb(0, 116, 255)',
 'rgb(0, 121, 255)',
 'rgb(0, 125, 255)',
 'rgb(0, 129, 255)',
 'rgb(0, 133, 255)',
 'rgb(0, 137, 255)',
 'rgb(0, 142, 255)',
 'rgb(0, 146, 255)',
 'rgb(0, 150, 255)',
 'rgb(0, 154, 255)',
 'rgb(0, 158, 255)',
 'rgb(0, 1

In [47]:
np.linspace(10, 1, 5)

array([10.  ,  7.75,  5.5 ,  3.25,  1.  ])

In [104]:
np.array(colorsys.hsv_to_rgb(250/360, 1, 1)) * 255

array([ 42.5,   0. , 255. ])